#Mounting Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
cd /content/drive/MyDrive/33project/05.AR/05.vid2vid_pytorch


/content/drive/MyDrive/33project/05.AR/05.vid2vid_pytorch


#Installing Libraries

In [4]:
!pip3 install imageio-ffmpeg
!pip install pyyaml
!pip3 install face-alignment
!pip3 install --upgrade --no-cache-dir gdown

#Importing Libraries

In [7]:
import matplotlib
matplotlib.use('Agg')
import yaml
from tqdm import tqdm
import imageio
import numpy as np

from skimage.transform import resize
from skimage import img_as_ubyte
import torch
import torch.nn.functional as F
from sync_batchnorm import DataParallelWithCallback
from modules.generator import OcclusionAwareGenerator, OcclusionAwareSPADEGenerator
from modules.keypoint_detector import KPDetector, HEEstimator
from animate import normalize_kp
from scipy.spatial import ConvexHull
import os, sys
from time import gmtime, strftime
from shutil import copy
from frames_dataset import FramesDataset
from modules.discriminator import MultiScaleDiscriminator
from train import train


#Defining Variables

In [8]:
weights='checkpoints'
config="config/vox-256-spade.yaml"
config_train="config/vox-256-spade.yaml"
checkpoint='checkpoints/15kp-ep119.pth.tar'
checkpoint_train=None

source_image='images/source_image_1.jpg'
driving_video='drivingVideo/sample_1.mp4'
result_video='result/video.mp4'

gen="original"
gen_infer="spade"

relative=True
adapt_scale=True

find_best_frame=True
best_frame=None

cpu=False
free_view=False
yaw=None
pitch=None
roll=None


mode="train"
log_dir="log"
device_ids=[0]

verbose=False
dataset_path='vox'

#Downloading PreTrained Weights

In [9]:
if os.path.exists(weights) is False:
  print("Downloading PreTrained Weights")
  !mkdir checkpoints
  %cd checkpoints
  !gdown https://drive.google.com/u/0/uc?id=1IMJyh7GumwQD2mlH77xA_UKPa7w0S_nR&export=download
  %cd ..
  !wget https://github.com/human-analysis/RankGAN/raw/master/models/hopenet_robust_alpha1.pkl -O checkpoints/hopenet_robust_alpha1.pkl


else:
  print('Weights Already Downloaded')

/content/drive/MyDrive/33project/05.AR/05.vid2vid_pytorch/checkpoints
Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1IMJyh7GumwQD2mlH77xA_UKPa7w0S_nR

but Gdown can't. Please check connections and permissions.
/content/drive/MyDrive/33project/05.AR/05.vid2vid_pytorch
--2026-09-28 07:38:30--  https://github.com/human-analysis/RankGAN/raw/master/models/hopenet_robust_alpha1.pkl
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/human-analysis/RankGAN/master/models/hopenet_robust_alpha1.pkl [following]
--2026-09-28 07:38:31--  https://raw.githubu

#Downloading Dataset
###Note : Size of Dataset is ~ 300GB. Sample from this Dataset has been provided to Train on. * Model has been trained on Multiple GPUS, training from scratch is not recommended.

In [10]:
if os.path.exists(dataset_path) is False:
  print("Downloading Dataset")

  !wget https://yt-dl.org/downloads/latest/youtube-dl -O youtube-dl
  !chmod a+rx youtube-dl
  !python load_videos.py --metadata vox-metadata.csv --format .mp4 --out_folder vox --workers 8
else:
  print('Dataset already Downloaded')

Dataset already Downloaded


#Training Script
###Please restart runtime before Training to release memory for Training

In [21]:
if sys.version_info[0] < 3:
    raise Exception("You must use Python 3 or higher. Recommended version is Python 3.7")

with open(config_train) as f:
    configg = yaml.load(f, Loader=yaml.FullLoader)

# 현재 프로젝트 위치에 맞게 경로 수정
configg['dataset_params']['root_dir'] = '/content/drive/MyDrive/33project/05.AR/05.vid2vid_pytorch/vox'
configg['train_params']['hopenet_snapshot'] = '/content/drive/MyDrive/33project/05.AR/05.vid2vid_pytorch/checkpoints/hopenet_robust_alpha1.pkl'

if gen == 'original':
    generator = OcclusionAwareGenerator(
        **configg['model_params']['generator_params'],
        **configg['model_params']['common_params']
    )
elif gen == 'spade':
    generator = OcclusionAwareSPADEGenerator(
        **configg['model_params']['generator_params'],
        **configg['model_params']['common_params']
    )

if torch.cuda.is_available():
    print('cuda is available')
    generator.to(device_ids[0])

if verbose:
    print(generator)

discriminator = MultiScaleDiscriminator(
    **configg['model_params']['discriminator_params'],
    **configg['model_params']['common_params']
)

if torch.cuda.is_available():
    discriminator.to(device_ids[0])

if verbose:
    print(discriminator)

kp_detector = KPDetector(
    **configg['model_params']['kp_detector_params'],
    **configg['model_params']['common_params']
)

if torch.cuda.is_available():
    kp_detector.to(device_ids[0])

if verbose:
    print(kp_detector)

he_estimator = HEEstimator(
    **configg['model_params']['he_estimator_params'],
    **configg['model_params']['common_params']
)

if torch.cuda.is_available():
    he_estimator.to(device_ids[0])

dataset = FramesDataset(
    is_train=(mode == 'train'),
    **configg['dataset_params']
)

if mode == 'train':
    print("Training...")
    train(
        configg,
        generator,
        discriminator,
        kp_detector,
        he_estimator,
        checkpoint_train,
        log_dir,
        dataset,
        device_ids
    )

cuda is available
Use predefined train-test split.
['id10195#4EhAfqcBJF0#000949#001089.mp4', 'id10964#EnvDKuGlPhU#005670#005850.mp4']
Training...
Loading hopenet


  0%|          | 0/300 [02:33<?, ?it/s]


KeyboardInterrupt: 

#Inference
### Restart runtime may be required. Final video would be saved in results folder.

In [26]:
# ============================================================
# Face-Vid2Vid Inference
# ============================================================

# 셀을 다시 실행해도 경로가 초기화되도록 맨 처음에 다시 지정
source_image = 'images/source_image_1.jpg'
driving_video = 'drivingVideo/sample_1.mp4'
result_video = 'result/video.mp4'


if sys.version_info[0] < 3:
    raise Exception(
        "You must use Python 3 or higher. Recommended version is Python 3.7"
    )


def load_checkpoints(config_path, checkpoint_path, gen, cpu=False):

    with open(config_path) as f:
        config = yaml.load(f, Loader=yaml.FullLoader)

    if gen_infer == 'original':
        generator = OcclusionAwareGenerator(
            **config['model_params']['generator_params'],
            **config['model_params']['common_params']
        )

    elif gen_infer == 'spade':
        generator = OcclusionAwareSPADEGenerator(
            **config['model_params']['generator_params'],
            **config['model_params']['common_params']
        )

    if not cpu:
        generator.cuda()

    kp_detector = KPDetector(
        **config['model_params']['kp_detector_params'],
        **config['model_params']['common_params']
    )

    if not cpu:
        kp_detector.cuda()

    he_estimator = HEEstimator(
        **config['model_params']['he_estimator_params'],
        **config['model_params']['common_params']
    )

    if not cpu:
        he_estimator.cuda()

    if cpu:
        checkpoint_data = torch.load(
            checkpoint_path,
            map_location=torch.device('cpu')
        )
    else:
        checkpoint_data = torch.load(checkpoint_path)

    generator.load_state_dict(checkpoint_data['generator'])
    kp_detector.load_state_dict(checkpoint_data['kp_detector'])
    he_estimator.load_state_dict(checkpoint_data['he_estimator'])

    if not cpu:
        generator = DataParallelWithCallback(generator)
        kp_detector = DataParallelWithCallback(kp_detector)
        he_estimator = DataParallelWithCallback(he_estimator)

    generator.eval()
    kp_detector.eval()
    he_estimator.eval()

    return generator, kp_detector, he_estimator


def headpose_pred_to_degree(pred):

    device = pred.device

    idx_tensor = [idx for idx in range(66)]
    idx_tensor = torch.FloatTensor(idx_tensor).to(device)

    pred = F.softmax(pred, dim=1)

    degree = torch.sum(
        pred * idx_tensor,
        axis=1
    ) * 3 - 99

    return degree


def get_rotation_matrix(yaw, pitch, roll):

    yaw = yaw / 180 * 3.14
    pitch = pitch / 180 * 3.14
    roll = roll / 180 * 3.14

    roll = roll.unsqueeze(1)
    pitch = pitch.unsqueeze(1)
    yaw = yaw.unsqueeze(1)

    pitch_mat = torch.cat([
        torch.ones_like(pitch),
        torch.zeros_like(pitch),
        torch.zeros_like(pitch),

        torch.zeros_like(pitch),
        torch.cos(pitch),
        -torch.sin(pitch),

        torch.zeros_like(pitch),
        torch.sin(pitch),
        torch.cos(pitch)
    ], dim=1)

    pitch_mat = pitch_mat.view(
        pitch_mat.shape[0],
        3,
        3
    )

    yaw_mat = torch.cat([
        torch.cos(yaw),
        torch.zeros_like(yaw),
        torch.sin(yaw),

        torch.zeros_like(yaw),
        torch.ones_like(yaw),
        torch.zeros_like(yaw),

        -torch.sin(yaw),
        torch.zeros_like(yaw),
        torch.cos(yaw)
    ], dim=1)

    yaw_mat = yaw_mat.view(
        yaw_mat.shape[0],
        3,
        3
    )

    roll_mat = torch.cat([
        torch.cos(roll),
        -torch.sin(roll),
        torch.zeros_like(roll),

        torch.sin(roll),
        torch.cos(roll),
        torch.zeros_like(roll),

        torch.zeros_like(roll),
        torch.zeros_like(roll),
        torch.ones_like(roll)
    ], dim=1)

    roll_mat = roll_mat.view(
        roll_mat.shape[0],
        3,
        3
    )

    rot_mat = torch.einsum(
        'bij,bjk,bkm->bim',
        pitch_mat,
        yaw_mat,
        roll_mat
    )

    return rot_mat


def keypoint_transformation(
    kp_canonical,
    he,
    estimate_jacobian=True,
    free_view=False,
    yaw=0,
    pitch=0,
    roll=0
):

    kp = kp_canonical['value']

    if not free_view:

        yaw, pitch, roll = (
            he['yaw'],
            he['pitch'],
            he['roll']
        )

        yaw = headpose_pred_to_degree(yaw)
        pitch = headpose_pred_to_degree(pitch)
        roll = headpose_pred_to_degree(roll)

    else:

        if yaw is not None:
            yaw = torch.tensor([yaw]).cuda()
        else:
            yaw = he['yaw']
            yaw = headpose_pred_to_degree(yaw)

        if pitch is not None:
            pitch = torch.tensor([pitch]).cuda()
        else:
            pitch = he['pitch']
            pitch = headpose_pred_to_degree(pitch)

        if roll is not None:
            roll = torch.tensor([roll]).cuda()
        else:
            roll = he['roll']
            roll = headpose_pred_to_degree(roll)

    t, exp = he['t'], he['exp']

    rot_mat = get_rotation_matrix(
        yaw,
        pitch,
        roll
    )

    # Keypoint rotation
    kp_rotated = torch.einsum(
        'bmp,bkp->bkm',
        rot_mat,
        kp
    )

    # Keypoint translation
    t = t.unsqueeze_(1).repeat(
        1,
        kp.shape[1],
        1
    )

    kp_t = kp_rotated + t

    # Expression deviation
    exp = exp.view(
        exp.shape[0],
        -1,
        3
    )

    kp_transformed = kp_t + exp

    if estimate_jacobian:

        jacobian = kp_canonical['jacobian']

        jacobian_transformed = torch.einsum(
            'bmp,bkps->bkms',
            rot_mat,
            jacobian
        )

    else:
        jacobian_transformed = None

    return {
        'value': kp_transformed,
        'jacobian': jacobian_transformed
    }


def make_animation(
    source_image,
    driving_video,
    generator,
    kp_detector,
    he_estimator,
    relative=True,
    adapt_movement_scale=True,
    estimate_jacobian=True,
    cpu=False,
    free_view=False,
    yaw=0,
    pitch=0,
    roll=0
):

    with torch.no_grad():

        predictions = []

        source = torch.tensor(
            source_image[np.newaxis].astype(np.float32)
        ).permute(
            0,
            3,
            1,
            2
        )

        if not cpu:
            source = source.cuda()

        driving = torch.tensor(
            np.array(
                driving_video
            )[np.newaxis].astype(np.float32)
        ).permute(
            0,
            4,
            1,
            2,
            3
        )

        print(source.shape)

        kp_canonical = kp_detector(source)

        he_source = he_estimator(source)

        he_driving_initial = he_estimator(
            driving[:, :, 0]
        )

        kp_source = keypoint_transformation(
            kp_canonical,
            he_source,
            estimate_jacobian
        )

        kp_driving_initial = keypoint_transformation(
            kp_canonical,
            he_driving_initial,
            estimate_jacobian
        )

        for frame_idx in tqdm(
            range(driving.shape[2])
        ):

            driving_frame = driving[
                :,
                :,
                frame_idx
            ]

            if not cpu:
                driving_frame = driving_frame.cuda()

            he_driving = he_estimator(
                driving_frame
            )

            kp_driving = keypoint_transformation(
                kp_canonical,
                he_driving,
                estimate_jacobian,
                free_view=free_view,
                yaw=yaw,
                pitch=pitch,
                roll=roll
            )

            kp_norm = normalize_kp(
                kp_source=kp_source,
                kp_driving=kp_driving,
                kp_driving_initial=kp_driving_initial,
                use_relative_movement=relative,
                use_relative_jacobian=estimate_jacobian,
                adapt_movement_scale=adapt_movement_scale
            )

            out = generator(
                source,
                kp_source=kp_source,
                kp_driving=kp_norm
            )

            predictions.append(
                np.transpose(
                    out[
                        'prediction'
                    ].data.cpu().numpy(),
                    [0, 2, 3, 1]
                )[0]
            )

    return predictions


def find_best_frame(
    source,
    driving,
    cpu=False
):

    import face_alignment

    def normalize_kp(kp):

        kp = kp - kp.mean(
            axis=0,
            keepdims=True
        )

        area = ConvexHull(
            kp[:, :2]
        ).volume

        area = np.sqrt(area)

        kp[:, :2] = (
            kp[:, :2] / area
        )

        return kp

    fa = face_alignment.FaceAlignment(
        face_alignment.LandmarksType._2D,
        flip_input=True,
        device='cpu' if cpu else 'cuda'
    )

    kp_source = fa.get_landmarks(
        255 * source
    )[0]

    kp_source = normalize_kp(
        kp_source
    )

    norm = float('inf')

    frame_num = 0

    for i, image in tqdm(
        enumerate(driving)
    ):

        kp_driving = fa.get_landmarks(
            255 * image
        )[0]

        kp_driving = normalize_kp(
            kp_driving
        )

        new_norm = (
            np.abs(
                kp_source - kp_driving
            ) ** 2
        ).sum()

        if new_norm < norm:
            norm = new_norm
            frame_num = i

    return frame_num


# ============================================================
# 입력 이미지 / 영상 불러오기
# ============================================================

source_image = imageio.imread(
    source_image
)

reader = imageio.get_reader(
    driving_video
)

fps = reader.get_meta_data()['fps']

driving_video_frames = []

try:

    for im in reader:
        driving_video_frames.append(im)

except RuntimeError:
    pass

reader.close()


source_image = resize(
    source_image,
    (256, 256)
)[..., :3]

driving_video_frames = [
    resize(
        frame,
        (256, 256)
    )[..., :3]
    for frame in driving_video_frames
]


# ============================================================
# Pretrained checkpoint 불러오기
# ============================================================

generator, kp_detector, he_estimator = load_checkpoints(
    config_path=config,
    checkpoint_path=checkpoint,
    gen=gen,
    cpu=cpu
)


# ============================================================
# Config 읽기
# ============================================================

with open(config_train) as f:

    config_data = yaml.load(
        f,
        Loader=yaml.FullLoader
    )


estimate_jacobian = config_data[
    'model_params'
][
    'common_params'
][
    'estimate_jacobian'
]

print(
    f'estimate jacobian: {estimate_jacobian}'
)


# ============================================================
# Animation 생성
# ============================================================

if find_best_frame or best_frame is not None:

    i = (
        best_frame
        if best_frame is not None
        else find_best_frame(
            source_image,
            driving_video_frames,
            cpu=cpu
        )
    )

    print(
        "Best frame: " + str(i)
    )

    driving_forward = (
        driving_video_frames[i:]
    )

    driving_backward = (
        driving_video_frames[:(i + 1)][::-1]
    )

    predictions_forward = make_animation(
        source_image,
        driving_forward,
        generator,
        kp_detector,
        he_estimator,
        relative=relative,
        adapt_movement_scale=adapt_scale,
        estimate_jacobian=estimate_jacobian,
        cpu=cpu,
        free_view=free_view,
        yaw=yaw,
        pitch=pitch,
        roll=roll
    )

    predictions_backward = make_animation(
        source_image,
        driving_backward,
        generator,
        kp_detector,
        he_estimator,
        relative=relative,
        adapt_movement_scale=adapt_scale,
        estimate_jacobian=estimate_jacobian,
        cpu=cpu,
        free_view=free_view,
        yaw=yaw,
        pitch=pitch,
        roll=roll
    )

    predictions = (
        predictions_backward[::-1]
        + predictions_forward[1:]
    )

else:

    predictions = make_animation(
        source_image,
        driving_video_frames,
        generator,
        kp_detector,
        he_estimator,
        relative=relative,
        adapt_movement_scale=adapt_scale,
        estimate_jacobian=estimate_jacobian,
        cpu=cpu,
        free_view=free_view,
        yaw=yaw,
        pitch=pitch,
        roll=roll
    )


# ============================================================
# 결과 영상 저장
# ============================================================

imageio.mimsave(
    result_video,
    [
        img_as_ubyte(frame)
        for frame in predictions
    ],
    fps=fps
)

print("완료!")
print("결과 영상:", result_video)

/tmp/ipykernel_1185/607957491.py:464: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  source_image = imageio.imread(


FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/15kp-ep119.pth.tar'